# Text Summarization using Seq2Seq with Attention

This notebook provides a clean, modular reproduction of the supplied **abstractive text summarization** project. It uses an Encoder-Decoder LSTM, additive attention, teacher forcing during training, and token-by-token inference.

> **Responsible use:** This is an educational portfolio demonstration trained on a small deterministic synthetic corpus. Generated summaries may be incomplete or inaccurate and require human review. Do not use private, legal, medical, financial, safety-critical, or official documents.

In [ ]:
import os
os.environ.setdefault("KERAS_BACKEND", "jax")

from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MODEL_METADATA_PATH, MODEL_METRICS_PATH, SAMPLE_ARTICLES_PATH
from src.data_preprocessing import generate_summarization_dataset, prepare_dataset, split_dataset
from src.model_evaluation import compute_rouge
from src.summarization_inference import Summarizer
from src.text_preprocessing import clean_text
from src.visualization import create_attention_heatmap

print("Project root:", PROJECT_ROOT)

## 1. Dataset reconstruction

The uploaded notebook used a deterministic synthetic fallback dataset with `article` and `summary` columns. It contains 2,500 organization/action/theme/impact examples and no private or copyrighted documents.

In [ ]:
dataset = prepare_dataset(generate_summarization_dataset(n=2500, seed=42))
train_df, val_df, test_df = split_dataset(dataset, seed=42)
print("Full:", dataset.shape)
print("Train / validation / test:", train_df.shape, val_df.shape, test_df.shape)
dataset[["article", "summary"]].head(3)

## 2. Preprocessing and Seq2Seq targets

- Text is lowercased.
- HTML and non-alphanumeric punctuation are removed.
- Repeated whitespace is normalized.
- Target summaries receive `sostok` and `eostok` boundary markers.
- Source sequences are post-padded to 49 tokens.
- Target sequences are post-padded to 12 tokens.
- Decoder targets are shifted one token ahead of decoder inputs.

In [ ]:
example = dataset.iloc[0]
print("Original article:
", example["article"])
print("
Clean article:
", example["article_clean"])
print("
Decoder sequence:
", example["summary_seq"])

## 3. Model architecture

The encoder LSTM returns token-level representations and final hidden/cell states. The decoder LSTM uses those states, while additive attention creates a context vector from the encoder outputs at every generated token. A time-distributed softmax layer predicts the next summary token.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(PROJECT_ROOT / "outputs/model_architecture.png")))

## 4. Load verified pretrained artifacts

The Streamlit application loads separate encoder and decoder inference models and does not retrain during startup.

In [ ]:
summarizer = Summarizer()
metadata = json.loads(MODEL_METADATA_PATH.read_text(encoding="utf-8"))
print("Training model parameters:", metadata["architecture"]["training_model_parameters"])
print("Source vocabulary:", metadata["sequence"]["source_vocab_size"])
print("Target vocabulary:", metadata["sequence"]["target_vocab_size"])

## 5. Generate a summary

Greedy decoding selects the highest-probability token at each step. Beam search is also implemented in the reusable inference module.

In [ ]:
samples = pd.read_csv(SAMPLE_ARTICLES_PATH)
sample = samples.iloc[0]
result = summarizer.summarize(sample["input_text"], decoding_method="greedy", include_attention=True)
print("INPUT:
", sample["input_text"])
print("
REFERENCE:
", sample["target_summary"])
print("
GENERATED:
", result.summary)
print("
Diagnostics:", result.as_record())

## 6. ROUGE evaluation for the sample

ROUGE-1 measures unigram overlap, ROUGE-2 measures bigram overlap, and ROUGE-L measures longest-common-subsequence overlap. These metrics should be paired with qualitative review.

In [ ]:
sample_rouge = compute_rouge(sample["target_summary"].lower().strip(" ."), result.summary)
pd.DataFrame([sample_rouge])

## 7. Attention alignment

The heatmap is generated from the trained Keras `AdditiveAttention` layer. It shows decoder alignment weights, not causal explanations.

In [ ]:
if result.attention_matrix is not None:
    figure = create_attention_heatmap(result.attention_matrix, result.source_tokens, result.generated_tokens)
    plt.show()

## 8. Verified held-out results and baselines

In [ ]:
metrics = json.loads(MODEL_METRICS_PATH.read_text(encoding="utf-8"))
model = metrics["seq2seq_attention"]
baselines = metrics["baselines"]
pd.DataFrame([
    {"Approach": "Lead sentence", **baselines["lead_sentence_baseline"]},
    {"Approach": "Lead 9 tokens", **baselines["lead_9_token_baseline"]},
    {"Approach": "Seq2Seq + Attention", "rouge_1_f1": model["rouge_1_f1"], "rouge_2_f1": model["rouge_2_f1"], "rouge_l_f1": model["rouge_l_f1"]},
])

## 9. Optional retraining

Run the following from the project root to reproduce training on the deterministic synthetic corpus:

```bash
python train_model.py --samples 2500 --epochs 12 --batch-size 64 --output-dir models/retrained
```

Retraining writes separate artifacts under `models/retrained/` so the supplied verified models are preserved.

## Limitations

- The training corpus is small and templated.
- The source vocabulary contains only 87 learned words plus padding.
- Inputs beyond 49 cleaned tokens are truncated.
- Out-of-domain text may produce generic or incorrect summaries.
- LSTM Seq2Seq models are less capable than modern Transformer and LLM summarizers for long, diverse documents.
- Generated summaries require human review.